---
title: Глава 7. Генерация, обработка и преобразование данных
subtitle: SQL Lab in JupyterLab
# license: CC-BY-4.0
github: https://github.com/magus1968/learning-sql
subject: Technical Portfolio
venue: GitHub & GitVerse Pages
# abstract: |
#   В последнем запросе главы, в разделе _Этот таинственный null_, увидим причину, по которой для усечения строк в дальнейшем будет использоваться Pandas.
authors:
  - name: Alex Smirnov
    email: a@smirnovs.pro
    corresponding: true
    affiliations: Data & BI Analyst
date: 2026-08-17
abbreviations:
    MyST: Markedly Structured Text
    Jupyter Book: Build static Web-books
    JupySQL: Run & highlight SQL in Jupyter
---

In [1]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL


connection_url = URL.create(
    drivername="mysql+pymysql",
    host="localhost",
    port=3306,
    database="sakila",
    username="root",
    password="********",
)

engine = create_engine(connection_url)

%load_ext sql

%config SqlMagic.displaylimit = 20
%config SqlMagic.displaycon = False
# %config SqlMagic.feedback = False

%sql engine

print("SQLAlchemy - подключение создано")
print("JupySQL - успешно подключен через SQLAlchemy Engine!")

SQLAlchemy - подключение создано
JupySQL - успешно подключен через SQLAlchemy Engine!


## Работа со строковыми данными

При работе со строковыми данными используется один из символьных типов данных: `char`, `varchar`, `text` (tinytext, text, mediumtext, longtext).

In [2]:
%%sql
CREATE TABLE string_tbl
    (char_fld CHAR(30),
    vchar_fld VARCHAR(30),
    text_fld TEXT
);

++
||
++
++

### Генерация строк

Самый простой способ заполнить символьный столбец – заключить строку в кавычки.

In [26]:
%%sql
INSERT INTO string_tbl (char_fld, vchar_fld, text_fld)
VALUES ('This is char data',
    'This is varchar data',
    'This is text data');

1 rows affected.

++
||
++
++

In [27]:
%%sql
SELECT * FROM string_tbl;

1 rows affected.

char_fld,vchar_fld,text_fld
This is char data,This is varchar data,This is text data


:::{warning}

При вставке строковых данных: если длина строки превышает максимальный размер для символьного столбца, сервер сгенерирует **исключение** (exception) – сигнал о возникновении аварийной ситуации, которая нарушает нормальный ход выполнения программы.
:::

Попытаемся заменить столбец vchar_fld строкой в 38 символов:

In [28]:
%%sql
UPDATE string_tbl
SET vchar_fld = 'This is a piece of extremely long data';

RuntimeError: (pymysql.err.DataError) (1406, "Data too long for column 'vchar_fld' at row 1")
[SQL: UPDATE string_tbl
SET vchar_fld = 'This is a piece of extremely long data';]
(Background on this error at: https://sqlalche.me/e/20/9h9h)


Получили исключение; данные остались не тронутыми:

In [29]:
%%sql
SELECT * FROM string_tbl;

1 rows affected.

char_fld,vchar_fld,text_fld
This is char data,This is varchar data,This is text data


Проверим в каком режиме работаем:

In [30]:
%%sql
SELECT @@session.sql_mode;

1 rows affected.

@@session.sql_mode
"ONLY_FULL_GROUP_BY,STRICT_TRANS_TABLES,NO_ZERO_IN_DATE,NO_ZERO_DATE,ERROR_FOR_DIVISION_BY_ZERO,NO_ENGINE_SUBSTITUTION"


Если предпочтем _**механизм усечения строк**_ вместо генерации исключений нужно выбрать режим ANSI:

In [31]:
%%sql
SET sql_mode = 'ansi';

++
||
++
++

In [32]:
%%sql
SELECT @@session.sql_mode;

1 rows affected.

@@session.sql_mode
"REAL_AS_FLOAT,PIPES_AS_CONCAT,ANSI_QUOTES,IGNORE_SPACE,ONLY_FULL_GROUP_BY,ANSI"


Вызвав инструкцию `UPDATE` повторно, обнаружим что сервер выполнит ее без ошибок и обновит столбец:

In [33]:
%%sql
UPDATE string_tbl
SET vchar_fld = 'This is a piece of extremely long data';

1 rows affected.

++
||
++
++

:::{attention} Однако внутри сессии будет зафиксировано скрытое предупреждение
Чтобы принудительно запросить у сервера текст предупреждения, необходимо следующим шагом выполнить команду `SHOW WARNINGS`, чтобы вывести диагностический результат последней выполненной операции.

Получим предупреждение о принудительном усечении текста под максимальный размер столбца: *В первой строке таблицы (at row 1) вы попытались записать в столбец vchar_fld слишком длинный текст. База данных не стала аварийно прерывать работу, но физически отрезала (truncated) лишний хвост строки, который не поместился в лимит столбца, и сохранила только разрешенную часть*.

In [34]:
%%sql
SHOW WARNINGS;

1 rows affected.

Level,Code,Message
Warning,1265,Data truncated for column 'vchar_fld' at row 1


Выполнив выборку столбца *vchar_fld*, увидим что строка действительно была усечена:

In [35]:
%%sql
SELECT vchar_fld
FROM string_tbl;

1 rows affected.

vchar_fld
This is a piece of extremely l


Чтобы вернуть настройки обратно в **безопасное состояние** с включенным **строгим контролем данных**, сбрасываем их в значение по умолчанию:

In [36]:
%%sql
SET sql_mode = DEFAULT;

++
||
++
++

In [37]:
%%sql
SELECT @@session.sql_mode;

1 rows affected.

@@session.sql_mode
"ONLY_FULL_GROUP_BY,STRICT_TRANS_TABLES,NO_ZERO_IN_DATE,NO_ZERO_DATE,ERROR_FOR_DIVISION_BY_ZERO,NO_ENGINE_SUBSTITUTION"


:::{hint} Лучший способ избежать исключений или усечения строки

При работе со столбцами `varchar` установить для верхнего порога достаточно высокое значение, которого хватит для обработки самых длинных строк, помещаемых в столбец.

Сервер выделяет для хранения такой строки только необходимое место, поэтому установка высокого предела для столбцов `varchar` не является расточительной.
:::

---

#### Включение одинарных кавычек

Поскольку строки указываются одинарными кавычками, следует обратить особое внимание на строки, содержащие внутри одинарные кавычки или апострофы.

In [38]:
%%sql
UPDATE string_tbl
SET text_fld = 'This string dosn't work';

RuntimeError: If using snippets, you may pass the --with argument explicitly.
For more details please refer: https://jupysql.ploomber.io/en/latest/compose.html#with-argument


Original error message from DB driver:
(pymysql.err.ProgrammingError) (1064, "You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near 't work'' at line 2")
[SQL: UPDATE string_tbl
SET text_fld = 'This string dosn't work';]
(Background on this error at: https://sqlalche.me/e/20/f405)



Чтобы серевер игнорировал апостроф, необходимо добавить перед ним управляющий символ `'` или символ обратной косой черты `\`

In [39]:
%%sql
UPDATE string_tbl
SET text_fld = 'This string didn''t work, but it does now';

1 rows affected.

++
||
++
++

In [40]:
%%sql
UPDATE string_tbl
SET text_fld = 'This string didn\'t work, but it does now';

1 rows affected.

++
||
++
++

In [41]:
%%sql
SELECT text_fld
FROM string_tbl;

1 rows affected.

text_fld
"This string didn't work, but it does now"


Однако, если извлекаете строку для добавления в файл (который будет читать другая программа), то может потребоваться включение управляющего символа как части извлеченной строки.

:::{attention} Функция `quote()`
:class: simple
:icon: false
Заключает всю строку в кавычки и добавляет управляющие символы к любым одинарным кавычкам или апострофам внутри строки.
:::

In [42]:
%%sql
SELECT QUOTE(text_fld)
FROM string_tbl;

1 rows affected.

QUOTE(text_fld)
"'This string didn\'t work, but it does now'"


При получении данных для экспорта целесообразно использовать функцию `quote()` для всех символьных столбцов не генерируемых системой – которые пользователи заполняют вручную на обычном языке (комментарии, описания, адреса, отзывы). В них может встретиться любой неожиданный символ.

:::{hint} Алан Бьюли дает отличный совет для реальной работы

Eсли выгружаете базу данных в текстовый файл для передачи аналитикам или в другую систему, всегда пропускайте текстовые комментарии через `quote()`. Это гарантирует, что файл откроется в Excel идеально ровно.
:::

---

#### Включение специальных символов

Если приложение является многонациональным, в нем, вероятно, будут встречаться строки, содержащие символы, которых нет на клавиатуре и может потребоваться включить такие символы, например, с диакритическими знаками **é** или **ö**.

:::{attention} Функция `char()`
:class: simple
:icon: false
Позволяет создавать строки, содержащие любые из 255 символов в наборе символов ASCII.
:::

In [5]:
%%sql
SELECT 'abcdefg', CHAR(97, 98, 99, 100, 101, 102, 103);

1 rows affected.

abcdefg,"CHAR(97, 98, 99, 100, 101, 102, 103)"
abcdefg,b'abcdefg'


:::{note} Разница в выводе MySQL CLI vs Jupyter

Функция `CHAR()` берет числа (ASCII-коды) и возвращает соответствующую им последовательность байт. По умолчанию в MySQL результат функции `CHAR()` возвращается как бинарная строка (тип данных `BINARY` или `BLOB`), а не как обычный читаемый текст (`VARCHAR`).

Алан Бьюли выполняет запросы в стандартной консоли MySQL CLI (Command Line Client), которая автоматически преобразует бинарные байты в читаемые символы на экране. Она не пытается показать типы данных Python, поэтому у автора в книге вывелся чистый текст `abcdefg`.

Когда Jupyter через Python-драйвер делает запрос к базе данных, он получает от MySQL сырой бинарный поток байт. Современные библиотеки Python (особенно при работе через JupySQL) честно предупреждают: _Внимание, база данных вернула этот текст в виде чистых байт, а не текстовой строки_ – и добавляют маркер b (binary) `b'abcdefg'`.
:::

:::{hint} Чтобы в Jupyter получить чистый текст 

Чтобы MySQL принудительно вернул результат как обычную текстовую строку (а не бинарную), можно явно указать кодировку с помощью ключевого слова `USING` внутри функции `CHAR(97, 98, 99 USING utf8mb4)`
:::

In [6]:
%%sql
SELECT 'abcdefg', CHAR(97, 98, 99, 100, 101, 102, 103 USING utf8mb4);

1 rows affected.

abcdefg,"CHAR(97, 98, 99, 100, 101, 102, 103 USING utf8mb4)"
abcdefg,abcdefg


:::{hint} Чтобы Jupyter вывел те же буквы расширенной латиницы

Что и в оригинале у Алана Бьюли `Çüéâäàåçêë`, нужно использовать кодировку `binary` для функции `CHAR()` и затем перекодировать её в `cp850`.

В оригинальной консоли MySQL у автора использовалась старая DOS-кодировка Western European (`cp850`). Именно в ней числа 128–137 будут теми же буквами что и у автора.
:::

In [26]:
%%sql
SELECT CONVERT(
  CHAR(128, 129, 130, 131, 132, 133, 134, 135, 136, 137 USING binary) 
  USING cp850
) decoded_result;

1 rows affected.

decoded_result
Çüéâäàåçêë


- `CHAR(... USING binary)` заставляет MySQL сгенерировать чистые, неиспорченные байты
- `CONVERT(... USING cp850)` принудительно расшифровывает их по старой западной DOS-таблице автора книги, превращая в правильные буквы Çüéâäàåçêë.
- Драйвер Python видит этот готовый текст и выводит его в Jupyter идеально чисто, без маркера `b` и без кракозябр.

In [30]:
%%sql
SELECT CONVERT(
    CHAR(138, 139, 140, 141, 142, 143, 144, 145, 146, 147 USING binary)
    USING cp850
) decoded_result;

1 rows affected.

decoded_result
èïîìÄÅÉæÆô


In [31]:
%%sql
SELECT CONVERT(
    CHAR(148, 149, 150, 151, 152, 153, 154, 155, 156, 157 USING binary)
    USING cp850
) decoded_result;

1 rows affected.

decoded_result
öòûùÿÖÜø£Ø


In [32]:
%%sql
SELECT CONVERT(
    CHAR(158, 159, 160, 161, 162, 163, 164, 165 USING binary)
    USING cp850
) decoded_result;

1 rows affected.

decoded_result
×ƒáíóúñÑ


Используем функцию `concat()` для соединения отдельных строк, одни из которых просто введем с клавиатуры, а другие сгенерируем с помощью `char()`:

In [37]:
%%sql
SELECT CONCAT(
  'danke sch', 
  CONVERT(CHAR(148 USING binary) USING cp850), 
  'n'
) result;

1 rows affected.

result
danke schön


:::{attention} Функция `ascii()`
:class: simple
:icon: false
Возвращает номер (эквивалент в ASCII) крайнего слева символа в переданной строке.
:::

In [38]:
%%sql
SELECT ASCII('ö');

1 rows affected.

ASCII('ö')
195


:::{note} Почему `ASCII('ö')` возвращает 195 вместо 148 (как в книге):

В современном UTF-8 (utf8mb4) буква `ö` состоит из двух байт: 195 и 182. Функция ASCII() считывает только самый первый байт строки, поэтому возвращает 195.

В книге автора использовалась старая однобайтовая кодировка CP850, где буква `ö` кодируется одним единственным байтом со значением 148.

**Решение для воссоздания книжного результата**: чтобы получить 148, нужно принудительно перевести символ в кодировку автора
```sql
SELECT ASCII(CONVERT('ö' USING cp850));
```
:::

In [39]:
%%sql
SELECT ASCII(CONVERT('ö' USING cp850)) AS book_result;

1 rows affected.

book_result
148


Используя функции `char()`, `ascii()` и `concat()` можно справиться с любой латиницей, даже если под рукой клавиатура без диакритических знаков и специальных символов.

---

### Манипуляции строками

Каждый сервер данных включает множество встроенных функций для манипуляции строками.

Прежде чем продолжить, сбросим и обновим данные в _string_tbl_:

In [43]:
%%sql
DELETE FROM string_tbl;

1 rows affected.

++
||
++
++

In [44]:
%%sql
INSERT INTO string_tbl (char_fld, vchar_fld, text_fld)
VALUES ('This string is 28 characters',
    'This string is 28 characters',
    'This string is 28 characters');

1 rows affected.

++
||
++
++

In [45]:
%%sql
SELECT * FROM string_tbl;

1 rows affected.

char_fld,vchar_fld,text_fld
This string is 28 characters,This string is 28 characters,This string is 28 characters



---

#### Строковые функции, возвращающие числовые значения

:::{attention} Функция `length()`
:class: simple
:icon: false
Возвращает количество символов в строке:
:::

In [46]:
%%sql
SELECT LENGTH(char_fld) char_length,
  LENGTH(vchar_fld) varchar_length,
  LENGTH(text_fld) text_length
FROM string_tbl;

1 rows affected.

char_length,varchar_length,text_length
28,28,28


Увидим одни и те же результаты для всех строковых функций независимо от типа стобца, в котором хранятся строки.

:::{attention} Функция `position()`
:class: simple
:icon: false
Находит местоположение подстроки внутри строки:
:::

In [47]:
%%sql
SELECT POSITION('characters' IN vchar_fld)
FROM string_tbl;

1 rows affected.

POSITION('characters' IN vchar_fld)
19


Если подстрока не может быть найдена, функция `position ()` возвращает значение 0.

:::{attention}
При работе с базами данных следует помнить, что первый символ строки находится в позиции 1.

Возвращаемое значение 0 указывает, что подстрока не может быть найдена, а не то, что подстрока является началом строки в которой выполняется поиск.
:::

Нестандартная функция `locate()` подобна функции `position()` но допускает необязательный третий параметр, используемый для указания начальной позиции поиска:

In [48]:
%%sql
SELECT LOCATE('is', vchar_fld, 5)
FROM string_tbl;

1 rows affected.

"LOCATE('is', vchar_fld, 5)"
13


:::{attention} Функция сравнения строк `strcmp()`
:class: simple
:icon: false
Принимает в качестве аргументов две строки и возвращает одно из значений:
- `-1`, если первая строка предшествует второй в порядке сортировки;
- ` 0`, если строки идентичны;
- ` 1`, если первая строка в порядке сортировки идет после второй.

**Нечувствительна** к регистру.
:::

In [49]:
%%sql
DELETE FROM string_tbl;

1 rows affected.

++
||
++
++

Сначала посмотрим порядок сортировки пяти строк используя запрос, а затем как строки сравниваются одна с другой, используя `strcmp()`.

In [50]:
%%sql
INSERT INTO string_tbl (vchar_fld)
VALUES ('abcd'),
       ('xyz'),
       ('QRSTUV'),
       ('qrstuv'),
       ('12345');

5 rows affected.

++
||
++
++

In [51]:
%%sql
SELECT vchar_fld
FROM string_tbl
ORDER BY vchar_fld;

5 rows affected.

vchar_fld
12345
abcd
QRSTUV
qrstuv
xyz


In [52]:
%%sql
SELECT STRCMP('12345', '12345') 12345_12345,
  STRCMP('abcd', 'xyz') abcd_xyz,
  STRCMP('abcd', 'QRSTUV') abcd_QRSTUV,
  STRCMP('qrstuv', 'QRSTUV') qrstuv_QRSTUV,
  STRCMP('12345', 'xyz') 12345_xyz,
  STRCMP('xyz', 'qrstuv') xyz_qrstuv;

1 rows affected.

12345_12345,abcd_xyz,abcd_QRSTUV,qrstuv_QRSTUV,12345_xyz,xyz_qrstuv
0,-1,-1,0,-1,1


Наряду с функцией `strcmp()` MySQL позволят использовать для сравнения строк в предложении **select** операторы `like` и `regexp`.

Такие сравнения будут давать значения 1 (истина) или 0 (ложь).

In [53]:
%%sql
SELECT name, name LIKE '%y' ends_in_y
FROM category;

16 rows affected.

name,ends_in_y
Action,0
Animation,0
Children,0
Classics,0
Comedy,1
Documentary,1
Drama,0
Family,1
Foreign,0
Games,0


In [55]:
%%sql
SELECT name, name REGEXP 'y$' ends_in_y
FROM category;

16 rows affected.

name,ends_in_y
Action,0
Animation,0
Children,0
Classics,0
Comedy,1
Documentary,1
Drama,0
Family,1
Foreign,0
Games,0



---

#### Строковые функции, возвращающие строки

В некоторых случаях требуется изменять существующие строки, извлекая часть строки или добавляя к ней дополнительный текст.

In [54]:
%%sql
DELETE FROM string_tbl;

5 rows affected.

++
||
++
++

In [55]:
%%sql
INSERT INTO string_tbl (text_fld)
VALUES ('This string was 29 characters');

1 rows affected.

++
||
++
++

:::{attention} Функция `concat()`
:class: simple
:icon: false
Полезна во многих ситуациях.
:::

В том числе когда нужно добавить к хранимой строке дополнительные символы:

In [56]:
%%sql
UPDATE string_tbl
SET text_fld = CONCAT(text_fld, 'but now it is longer');

1 rows affected.

++
||
++
++

In [57]:
%%sql
SELECT text_fld
FROM string_tbl;

1 rows affected.

text_fld
This string was 29 charactersbut now it is longer


Еще одно распространенное использование функции `concat()` – построение строки из отдельных фрагментов данных.

In [58]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

print(f'Pandas ver. {pd.__version__}')
print('Ограничение на макс. ширину столбцов в 50 символов отключено.')

Pandas ver. 3.0.5
Ограничение на макс. ширину столбцов в 50 символов отключено.


In [59]:
%config SqlMagic.autopandas = True

In [60]:
%%sql
SELECT CONCAT(first_name, ' ', last_name,
    ' has been a customer since ',
    date(create_date)) cust_narrative
FROM customer;

599 rows affected.

,cust_narrative
0,MARY SMITH has been a customer since 2006-02-14
1,PATRICIA JOHNSON has been a customer since 2006-02-14
2,LINDA WILLIAMS has been a customer since 2006-02-14
3,BARBARA JONES has been a customer since 2006-02-14
4,ELIZABETH BROWN has been a customer since 2006-02-14
...,...
594,TERRENCE GUNDERSON has been a customer since 2006-02-14
595,ENRIQUE FORSYTHE has been a customer since 2006-02-14
596,FREDDIE DUGGAN has been a customer since 2006-02-14
597,WADE DELVALLE has been a customer since 2006-02-14


Функция `concat()` может обрабатывать любое выражение, возвращающее строку, и даже преобразовывать числа и даты в строковый формат, о чем свидетельствует столбец даты (create_date), использованный в качестве аргумента.

---

In [61]:
%config SqlMagic.autopandas = False

Хотя функция `concat()` полезна для добавления символов в начало или конец строки, может потребоваться добавить или заменить символы в середине строки.

:::{attention} Функция `insert()`
:class: simple
:icon: false
Принимает четыре аргумента:
- исходную строку,
- позицию, с которой следует начинать вставку,
- количество вставляемых символов,
- вставляемую строку.

В зависимости от значения третьего аргумента функция может использоваться как для вставки, так и для замены символов в строке. \
При значении третьего аргумента `0` строка вставляется (любые завершающие символы сдвигаются вправо):

In [62]:
%%sql
SELECT INSERT('goodbye world', 9, 0, 'cruel ') string;

1 rows affected.

string
goodbye cruel world


Если третий аргумент больше нуля, то он указывает количество символов, которые заменяются вставляемой строкой:

In [63]:
%%sql
SELECT INSERT('goodbye world', 1, 7, 'hello') string;

1 rows affected.

string
hello world


:::{attention} Функция `replace()`
:class: simple
:icon: false
Предназначена для замены одной подстроки другой.

Заменяет _каждый_ экземпляр искомой подстроки заменяемой строкой, поэтому нужно быть осторожным, чтобы не получить больше замен, чем ожидается.
:::

In [64]:
%%sql
SELECT REPLACE('goodbye world goodbye', 'goodbye', 'hello') string;

1 rows affected.

string
hello world hello


Помимо вставки символов в строку, может потребоваться извлечение подстроки из строки.

:::{attention} Функция `substring()`
:class: simple
:icon: false
Извлекает указанное количество символов, начиная с заданной позиции.
:::

In [65]:
%%sql
SELECT SUBSTRING('goodbye cruel world', 9, 5);

1 rows affected.

"SUBSTRING('goodbye cruel world', 9, 5)"
cruel



---

## Работа с числовыми данными

В отличие от строковых и временн*ы*х данных, генерация числовых данных довольно проста. Можете ввести число, получить его из другого столбца или сгенерировать его путем вычислений. При выполнении вычислений доступны все обычные арифметические операторы, а для изменения приоритетов вычислений можно использовать скобки.

In [66]:
%%sql
SELECT (37 * 59) / (78 - (8 * 6));

1 rows affected.

(37 * 59) / (78 - (8 * 6))
72.7667


Основная проблема при хранении числовых данных заключается в том, что числа могут быть округлены, если они больше размера, указанного для числового столбца. Например, число 9,96 будет округлено до 10,0 если оно сохранено в столбце определенном как `float(3,1)`.

### Выполнение математических функций

:::{note} Числовые функции от одного аргумента
:class: simple
:open: true
:icon: false

```text
| Имя функции | Описание                         |
| ----------- | -------------------------------- |
| acos(х)     | Вычисляет арккосинус х           |
| asin(х)     | Вычисляет арксинус х             |
| atan(х)     | Вычисляет арктангенс х           |
| cos(х)      | Вычисляет косинус х              |
| cot(х)      | Вычисляет котангенс х            |
| ехр(х)      | Вычисляет экспоненту х           |
| ln(x)       | Вычисляет натуральный логарифм х |
| sin(х)      | Вычисляет синус х                |
| sqrt(х)     | Вычисляет квадратный корень х    |
| tan(х)      | Вычисляет тангенс х              |
```
:::

:::{attention} Функция`mod()`
:class: simple
:icon: false
Оператор вычисления остатка от деления одного числа на другое.

В MySQL работает не только с целочисленными аргументами, но и с действительными числами.
:::

In [67]:
%%sql
SELECT MOD(10, 4);

1 rows affected.

"MOD(10, 4)"
2


In [68]:
%%sql
SELECT MOD(22.75, 5);

1 rows affected.

"MOD(22.75, 5)"
2.75


:::{attention} Функция `pow()`
:class: simple
:icon: false
Возвращает одно число возведенное в степень, равную значению второго числа.
:::

In [69]:
%%sql
SELECT POW(2, 8);

1 rows affected.

"POW(2, 8)"
256.0


In [70]:
%%sql
SELECT POW(2, 10) kilobyte, POW(2, 20) megabyte,
  POW(2, 30) gigabyte, POW(2, 40) terabyte;

1 rows affected.

kilobyte,megabyte,gigabyte,terabyte
1024.0,1048576.0,1073741824.0,1099511627776.0


### Управление точностью чисел

При работе с числами с плавающей точкой не всегда требуется работа или отображение числа с полной точностью. Например, можно хранить данежные данные транзакции с точностью до шести знаков после запятой, но для их отображения понадобится округление до ближайшей сотой.

:::{attention} Функции `ceil()` и `floor()`
:class: simple
:icon: false
Используются для округления в б*о*льшую или меньшую сторону к ближайшему целому.
:::

In [82]:
%%sql
SELECT CEIL(72.445), FLOOR(72.445);

1 rows affected.

CEIL(72.445),FLOOR(72.445)
73,72


In [81]:
%%sql
SELECT CEIL(72.000000001), FLOOR(72.999999999);

1 rows affected.

CEIL(72.000000001),FLOOR(72.999999999)
73,72


:::{attention} Функция `round()`
:class: simple
:icon: false
Округляет к ближайшему целому по обычным правилам.
:::

In [87]:
%%sql
SELECT ROUND(72.49999), ROUND(72.5), ROUND(72.50001);

1 rows affected.

ROUND(72.49999),ROUND(72.5),ROUND(72.50001)
72,73,73


В большинстве случаев требуется сохранить хотя бы некоторую часть десятичных знаков после запятой, а не округлять значение до целого числа.

Для этого функция `round()` допускает необязательный второй аргумент, который указывает какое количество цифр справа от запятой следует оставить при округлении.

In [88]:
%%sql
SELECT ROUND(72.0909, 1), ROUND(72.0909, 2), ROUND(72.0909, 3);

1 rows affected.

"ROUND(72.0909, 1)","ROUND(72.0909, 2)","ROUND(72.0909, 3)"
72.1,72.09,72.091


:::{attention} Функция `truncate()`
:class: simple
:icon: false
Отбрасывает _(усекает)_ ненужные цифры без округления.

Обязательный второй аргумент указывает количество цифр справа от десятичной запятой.
:::

In [90]:
%%sql
SELECT TRUNCATE(72.000000001, 0), TRUNCATE(72.999999999, 0);

1 rows affected.

"TRUNCATE(72.000000001, 0)","TRUNCATE(72.999999999, 0)"
72,72


In [91]:
%%sql
SELECT TRUNCATE(72.0909, 1), TRUNCATE(72.0909, 2), TRUNCATE(72.0909, 3);

1 rows affected.

"TRUNCATE(72.0909, 1)","TRUNCATE(72.0909, 2)","TRUNCATE(72.0909, 3)"
72.0,72.09,72.090


:::{note} Примечание
И `truncate()` и `round()` допускают *отрицательные* значения второго аргумента. Это означает, что усекаются или округляются цифры *слева* от десятичной точки.

Это поначалу может показаться странным, но для таких значений аргумента имеются вполне допустимые ситуации. Например, вы можете продавать продукт, который можно покупать только по 10 единиц. Если покупателем заказаны 17 единиц, можете выбрать один из спосовоб изменения количества товара в заказе клиента:
:::

In [94]:
%%sql
SELECT ROUND(17, -1), TRUNCATE(17, -1);

1 rows affected.

"ROUND(17, -1)","TRUNCATE(17, -1)"
20,10


### Работа со знаковыми данными

Если работаете с числовыми столбцами, в которых допускаются отрицательные значения, могут быть полезны некоторые специализированные числовые функции.

Допустим вас просят создать отчет, показывающий текущее состояние набора банковских счетов, используя данные из некой таблицы account:
```text
| account_id | acct_type    | balance |
| ---------- | ------------ | ------- |
| 123        | MONEY MARKET | 785.22  |
| 456        | SAVINGS      | 0.00    |
| 789        | CHECKING     | -324.22 |
```

Представленный далее запрос возвращает три столбца, полезных при создании отчета:
```sql
SELECT account_id, SIGN(balance), ABC(balance)
FROM account;
```
```text
| account_id | SIGN(balance) | ABC(balance) |
| ---------- | ------------- | ------------ |
| 123        | 1             | 785.22       |
| 456        | 0             | 0.00         |
| 789        | -1            | 324.22       |
```
Во втором столбце функция `sign()` возвращает значение -1, если баланс счета отрицательный, 0 если баланс нулевой и 1 если баланс положительный. Третий столбец получает абсолютное значение баланса счета с помощью функции `abc()`.

---

## Работа с временн*ы*ми данными

Из трех типов данных, обсуждаемых в этой главе (символьные, числовые и временн*ы*е), когда дело доходит до создания и обработки данных, временн*ы*е данные являются наиболее сложными.

Некоторая сложность вызвана множеством способов, которыми можно описать единственную дату и время. Б*о*льшая часть сложности связана с системой отсчета.

### Часовые пояса